In [ ]:
# Cell 1: Install dependencies
!pip install -q browserbase google-genai playwright python-dotenv pydantic
!playwright install chromium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 14.5 MB/s eta 0:00:00
170.4 MiB [] 0% 0.0s170.4 MiB [] 0% 100.0s170.4 MiB [] 0% 30.1s170.4 MiB [] 0% 22.2s170.4 MiB [] 0% 15.2s170.4 MiB [] 1% 7.8s170.4 MiB [] 2% 5.6s170.4 MiB [] 2% 4.7s170.4 MiB [] 3% 4.1s170.4 MiB [] 3% 4.3s170.4 MiB [] 4% 4.2s170.4 MiB [] 5% 3.8s170.4 MiB [] 6% 3.4s170.4 MiB [] 6% 3.2s170.4 MiB [] 7% 3.0s170.4 MiB [] 8% 2.9s170.4 MiB [] 9% 2.8s170.4 MiB [] 10% 2.7s170.4 MiB [] 11% 2.5s170.4 MiB [] 12% 2.4s170.4 MiB [] 13% 2.3s170.4 MiB [] 14% 2.2s170.4 MiB [] 15% 2.2s170.4 MiB [] 17% 2.1s170.4 MiB [] 18% 2.0s170.4 MiB [] 19% 2.0s170.4 MiB [] 19% 1.9s170.4 MiB [] 20% 1.9s170.4 MiB [] 21% 1.9s170.4 MiB [] 22% 1.8s170.4 MiB [] 23% 1.8s170.4 MiB [] 24% 1.7s170.4 MiB [] 25% 1.7s170.4 MiB [] 26% 1.6s170.4 MiB [] 27% 1.6s170.4 MiB [] 28% 1.6s170.4 MiB [] 29% 1.5s170.4 MiB [] 30% 1.5s170.4 MiB [] 32% 1.4s170.4 MiB [] 33% 1.4s170.4 MiB [] 

In [ ]:
# Cell 2: Load API keys from Colab Secrets
import os
from google.colab import userdata

# Load all 3 secrets
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["BROWSERBASE_API_KEY"] = userdata.get("BROWSERBASE_API_KEY")
os.environ["BROWSERBASE_PROJECT_ID"] = userdata.get("BROWSERBASE_PROJECT_ID")

# Verify they're loaded (without printing actual values)
print(" GEMINI_API_KEY loaded:", bool(os.environ.get("GEMINI_API_KEY")))
print(" BROWSERBASE_API_KEY loaded:", bool(os.environ.get("BROWSERBASE_API_KEY")))
print(" BROWSERBASE_PROJECT_ID loaded:", bool(os.environ.get("BROWSERBASE_PROJECT_ID")))

 GEMINI_API_KEY loaded: True
 BROWSERBASE_API_KEY loaded: True
 BROWSERBASE_PROJECT_ID loaded: True


In [ ]:
# Cell 3 — Clone team repo and import the browser class
import os, sys, subprocess

REPO_DIR = "/content/agent-recsys"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "-b", "feature",
         "https://github.com/akgupta-droid/Agent-Recsys.git",
         REPO_DIR],
        check=True,
    )

sys.path.insert(0, f"{REPO_DIR}/shopping_agent/tools")
from browserbase_product_browser import BrowserbaseProductBrowser, browse_products_with_browserbase

print(" Imported from", f"{REPO_DIR}/shopping_agent/tools/browserbase_product_browser.py")

 Imported from /content/agent-recsys/shopping_agent/tools/browserbase_product_browser.py


In [ ]:
# Cell 4 — Patch PRODUCT_URL_HINTS + register replacement sites

import browserbase_product_browser as bpb

# Patch 1: Add /pdp/ to PRODUCT_URL_HINTS
if "/pdp/" not in bpb.BrowserbaseProductBrowser.PRODUCT_URL_HINTS:
    bpb.BrowserbaseProductBrowser.PRODUCT_URL_HINTS = (
        bpb.BrowserbaseProductBrowser.PRODUCT_URL_HINTS + ["/pdp/"]
    )
print("PRODUCT_URL_HINTS patched:", bpb.BrowserbaseProductBrowser.PRODUCT_URL_HINTS)

# Patch 2: Register replacement sites (Article + Burrow as Wayfair substitutes)
REPLACEMENT_SITES = {
    "article": {
        "name": "Article",
        "search_url": "https://www.article.com/search?q={query}",
        "allowed_domains": ["article.com"],
    },
    "burrow": {
        "name": "Burrow",
        "search_url": "https://burrow.com/search?q={query}",
        "allowed_domains": ["burrow.com"],
    },
}

bpb.SITE_SEARCH_TEMPLATES.update(REPLACEMENT_SITES)
print("Registered replacement sites:", list(REPLACEMENT_SITES.keys()))
print("Total sites in config:", len(bpb.SITE_SEARCH_TEMPLATES))

PRODUCT_URL_HINTS patched: ['/p/', '/product', '/products', '/dp/', '/ip/', '/item', '/listing', '/pd/', 'sku=', 'pid=', 'productid', '/pdp/']
Registered replacement sites: ['article', 'burrow']
Total sites in config: 11


In [ ]:
# Cell 15 queries
QUERIES = [
    # Living room (5)
    {"id": "lr_01", "room": "living_room", "query": "modern minimalist fabric sofa",        "filters": {"max_price": 1200}},
    {"id": "lr_02", "room": "living_room", "query": "round oak coffee table mid-century",    "filters": {}},
    {"id": "lr_03", "room": "living_room", "query": "neutral wool area rug beige",            "filters": {}},
    {"id": "lr_04", "room": "living_room", "query": "arc floor lamp brass",                   "filters": {}},
    {"id": "lr_05", "room": "living_room", "query": "boucle accent chair cream",              "filters": {"max_price": 500}},
    # Dining room (5)
    {"id": "dr_01", "room": "dining_room", "query": "extendable wood dining table modern",    "filters": {}},
    {"id": "dr_02", "room": "dining_room", "query": "upholstered dining chairs beige linen",  "filters": {}},
    {"id": "dr_03", "room": "dining_room", "query": "wood sideboard buffet cabinet",          "filters": {}},
    {"id": "dr_04", "room": "dining_room", "query": "black pendant light dining room modern", "filters": {}},
    {"id": "dr_05", "room": "dining_room", "query": "washable dining room rug stain resistant","filters": {}},
    # Home office (5)
    {"id": "ho_01", "room": "home_office", "query": "L-shaped wood writing desk",             "filters": {}},
    {"id": "ho_02", "room": "home_office", "query": "ergonomic mesh office chair lumbar",     "filters": {"max_price": 400}},
    {"id": "ho_03", "room": "home_office", "query": "tall walnut bookshelf 5-shelf",          "filters": {}},
    {"id": "ho_04", "room": "home_office", "query": "adjustable LED desk lamp matte black",   "filters": {}},
    {"id": "ho_05", "room": "home_office", "query": "framed minimalist line art set of 3",    "filters": {}},
]
print(f"Total queries: {len(QUERIES)}")

Total queries: 15


In [ ]:
# Cell 6 — Run all 15 queries on 4 working US sites
import asyncio, csv, time, traceback, re
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

WORKING_SITES = ["west_elm", "amazon", "burrow", "article"]

OUT_CSV = Path("/content/browserbase_results.csv")
FIELDNAMES = [
    "query_id", "room", "query",
    "title", "product_url", "image_url", "description",
    "price_text", "price_num", "source_site", "category",
    "relevance_score", "why_it_matches",
    "passes_filter", "is_us_domain", "scraped_at",
]

US_DOMAINS = set()
for key in WORKING_SITES:
    if key in bpb.SITE_SEARCH_TEMPLATES:
        US_DOMAINS.update(bpb.SITE_SEARCH_TEMPLATES[key]["allowed_domains"])

print(f"Running {len(QUERIES)} queries on {WORKING_SITES}")
print(f"US domains: {sorted(US_DOMAINS)}")
print(f"Estimated time: ~{len(QUERIES) * 7} min\n")

call_kwargs = dict(
    country="US",
    max_results=20,
    max_product_pages=15,
    sites=WORKING_SITES,
    use_proxy=False,
)

def parse_price(price_text):
    if not price_text:
        return None
    m = re.search(r"[\d,]+\.?\d*", str(price_text).replace(",", ""))
    return float(m.group()) if m else None

def passes_price_filter(price_num, filters):
    if not filters or "max_price" not in filters:
        return True
    if price_num is None:
        return True
    return price_num <= filters["max_price"]

def is_us_domain(url):
    if not url:
        return False
    try:
        host = urlparse(url).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        return any(host == d or host.endswith("." + d) for d in US_DOMAINS)
    except Exception:
        return False

if OUT_CSV.exists():
    OUT_CSV.unlink()
with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()

total_start = time.time()
counts = []
non_us_dropped = 0

for i, q in enumerate(QUERIES, 1):
    print(f"[{i}/{len(QUERIES)}] {q['id']} | {q['room']} | {q['query']}")
    t0 = time.time()
    try:
        result = await browse_products_with_browserbase(q["query"], **call_kwargs)
        products = result.get("products", []) if isinstance(result, dict) else []
        notes = result.get("notes", []) if isinstance(result, dict) else []

        rows = []
        q_dropped = 0
        for r in products:
            url = r.get("product_url", "")
            if not is_us_domain(url):
                q_dropped += 1
                non_us_dropped += 1
                continue

            price_num = parse_price(r.get("price_text"))
            rows.append({
                "query_id":        q["id"],
                "room":            q["room"],
                "query":           q["query"],
                "title":           r.get("title", ""),
                "product_url":     url,
                "image_url":       r.get("image_url", ""),
                "description":     r.get("description", ""),
                "price_text":      r.get("price_text", ""),
                "price_num":       price_num if price_num is not None else "",
                "source_site":     r.get("source_site", ""),
                "category":        r.get("category", ""),
                "relevance_score": r.get("relevance_score", ""),
                "why_it_matches":  r.get("why_it_matches", ""),
                "passes_filter":   passes_price_filter(price_num, q.get("filters", {})),
                "is_us_domain":    True,
                "scraped_at":      datetime.now().isoformat(timespec="seconds"),
            })

        with OUT_CSV.open("a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDNAMES).writerows(rows)

        counts.append((q["id"], len(rows)))
        elapsed_q = time.time() - t0
        msg = f"  OK {len(rows)} products in {elapsed_q:.0f}s"
        if notes:
            msg += f"  | notes: {notes}"
        print(msg)

    except Exception as e:
        counts.append((q["id"], 0))
        print(f"  FAILED: {type(e).__name__}: {e}")

elapsed = time.time() - total_start
print(f"\n DONE in {elapsed/60:.1f} min")
print(f"CSV: {OUT_CSV}")
total = sum(n for _, n in counts)
print(f"Total products: {total}  |  avg {total/len(QUERIES):.1f} per query")
print(f"Non-US URLs dropped: {non_us_dropped}")
print("Per-query counts:", counts)

Running 15 queries on ['west_elm', 'amazon', 'burrow', 'article']
US domains: ['amazon.com', 'article.com', 'burrow.com', 'westelm.com']
Estimated time: ~105 min

[1/15] lr_01 | living_room | modern minimalist fabric sofa

[Browserbase Search Plan]
{
  "interpreted_need": "User is looking for a modern minimalist fabric sofa.",
  "room": null,
  "styles": [
    "modern",
    "minimalist"
  ],
  "categories": [
    "sofa"
  ],
  "constraints": [
    "fabric"
  ],
  "budget_text": null,
  "search_phrases": [
    "modern minimalist fabric sofa",
    "modern fabric sofa",
    "minimalist fabric sofa",
    "fabric sofa modern design",
    "minimalist couch fabric"
  ]
}
[Browserbase] Session started: https://browserbase.com/sessions/aa97997b-0aab-49d8-be2e-e230f1d92a1f
[Search] West Elm: modern minimalist fabric sofa
[Search] West Elm: modern fabric sofa
[Search] West Elm: minimalist fabric sofa
[Search] West Elm: fabric sofa modern design
[Search] Amazon: modern minimalist fabric sofa
[Sear

In [ ]:
# Cell — Fix product URLs
import pandas as pd

df = pd.read_csv("/content/browserbase_results.csv")
print(f"Total rows: {len(df)}")

def clean_url(url):
    if not isinstance(url, str):
        return url
    for sep in [',"data:', '","data:', ',data:']:
        if sep in url:
            url = url.split(sep)[0]
    url = url.rstrip(',/"\' ')
    return url

df['product_url'] = df['product_url'].apply(clean_url)
df.to_csv("/content/browserbase_results.csv", index=False)


Total rows: 99


In [ ]:
import pandas as pd
df = pd.read_csv("/content/browserbase_results.csv")

print(f"Total rows: {len(df)}")
print(f"Unique queries: {df['query_id'].nunique()} / 15\n")

print("Products per source site:")
print(df['source_site'].value_counts())

print("\nProducts per room:")
print(df['room'].value_counts())

print(f"\nRelevance scores:")
print(df['relevance_score'].describe().round(1))

print(f"\nPrice range:")
print(f"  Min: ${df['price_num'].min()}")
print(f"  Max: ${df['price_num'].max()}")
print(f"  Median: ${df['price_num'].median()}")

print(f"\nUS domain check: {df['is_us_domain'].all()}")
print(f"Empty descriptions: {(df['description'].isna() | (df['description'] == '')).sum()}")

Total rows: 99
Unique queries: 14 / 15

Products per source site:
source_site
West Elm    90
Amazon       9
Name: count, dtype: int64

Products per room:
room
dining_room    43
living_room    33
home_office    23
Name: count, dtype: int64

Relevance scores:
count     99.0
mean      71.9
std       23.7
min       10.0
25%       60.0
50%       75.0
75%       95.0
max      100.0
Name: relevance_score, dtype: float64

Price range:
  Min: $49.99
  Max: $3749.0
  Median: $599.0

US domain check: True
Empty descriptions: 0


In [ ]:
from google.colab import files
files.download("/content/browserbase_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
df = pd.read_csv("/content/browserbase_results.csv")

total = len(df)
real_images = df['image_url'].astype(str).str.startswith('http').sum()
base64_images = df['image_url'].astype(str).str.startswith('data:image').sum()
missing = df['image_url'].isna().sum()

print(f"Total products: {total}")
print(f"Real image URLs (http): {real_images} ({100*real_images/total:.0f}%)")
print(f"Base64 images: {base64_images} ({100*base64_images/total:.0f}%)")
print(f"Missing images: {missing} ({100*missing/total:.0f}%)")
print(f"Total problematic: {base64_images + missing} ({100*(base64_images+missing)/total:.0f}%)")

Total products: 99
Real image URLs (http): 70 (71%)
Base64 images: 5 (5%)
Missing images: 24 (24%)
Total problematic: 29 (29%)


In [ ]:
print(df[df['query_id']=='lr_05'][['title', 'price_num', 'passes_filter']])

                                           title  price_num  passes_filter
26  Cozy Swivel Chair (In-Stock & Ready To Ship)     799.00          False
27                         Crescent Swivel Chair     599.00          False
28                    Laurent Grand Swivel Chair     949.00          False
29                    Imogene Grand Swivel Chair    1379.00          False
30                            Haven Swivel Chair     624.99          False
31                           Berra Leather Chair    1399.00          False
32                   Lacon Leather Slipper Chair     559.20          False


In [ ]:
# Re-run only ho_05 with broader query
import asyncio, csv, time, re
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
import browserbase_product_browser as bpb

WORKING_SITES = ["west_elm", "amazon", "burrow", "article"]
OUT_CSV = Path("/content/browserbase_results.csv")

US_DOMAINS = set()
for key in WORKING_SITES:
    if key in bpb.SITE_SEARCH_TEMPLATES:
        US_DOMAINS.update(bpb.SITE_SEARCH_TEMPLATES[key]["allowed_domains"])

def is_us_domain(url):
    if not url: return False
    host = urlparse(url).netloc.lower()
    if host.startswith("www."): host = host[4:]
    return any(host == d or host.endswith("." + d) for d in US_DOMAINS)

def parse_price(p):
    if not p: return None
    m = re.search(r"[\d,]+\.?\d*", str(p).replace(",", ""))
    return float(m.group()) if m else None

# Broader query
q = {"id": "ho_05", "room": "home_office", "query": "framed minimalist wall art", "filters": {}}

print(f"Running {q['query']}...")
t0 = time.time()
result = await browse_products_with_browserbase(
    q["query"], country="US", max_results=20, max_product_pages=15,
    sites=WORKING_SITES, use_proxy=False
)
products = result.get("products", [])
print(f"Got {len(products)} products in {time.time()-t0:.0f}s")

# Append to existing CSV
new_rows = []
for r in products:
    url = r.get("product_url", "")
    if not is_us_domain(url): continue
    price_num = parse_price(r.get("price_text"))
    new_rows.append({
        "query_id": q["id"], "room": q["room"], "query": q["query"],
        "title": r.get("title", ""), "product_url": url,
        "image_url": r.get("image_url", ""), "description": r.get("description", ""),
        "price_text": r.get("price_text", ""),
        "price_num": price_num if price_num is not None else "",
        "source_site": r.get("source_site", ""), "category": r.get("category", ""),
        "relevance_score": r.get("relevance_score", ""),
        "why_it_matches": r.get("why_it_matches", ""),
        "passes_filter": True, "is_us_domain": True,
        "scraped_at": datetime.now().isoformat(timespec="seconds"),
    })

import pandas as pd
df_old = pd.read_csv(OUT_CSV)
df_new = pd.DataFrame(new_rows)
df_combined = pd.concat([df_old, df_new], ignore_index=True)
df_combined.to_csv(OUT_CSV, index=False)
print(f"Added {len(new_rows)} products to CSV")
print(f"Total now: {len(df_combined)} products, {df_combined['query_id'].nunique()} queries")

Running framed minimalist wall art...

[Browserbase Search Plan]
{
  "interpreted_need": "User is looking for framed minimalist wall art.",
  "room": null,
  "styles": [
    "minimalist"
  ],
  "categories": [
    "wall art",
    "framed art"
  ],
  "constraints": [
    "framed"
  ],
  "budget_text": null,
  "search_phrases": [
    "framed minimalist wall art",
    "minimalist framed art",
    "minimalist wall decor framed",
    "simple framed wall art",
    "modern minimalist framed art"
  ]
}
[Browserbase] Session started: https://browserbase.com/sessions/36d728b9-448e-4438-8529-69f5b8a7058c
[Search] West Elm: framed minimalist wall art
[Search] West Elm: minimalist framed art
[Search] West Elm: minimalist wall decor framed
[Search] West Elm: simple framed wall art
[Search] Amazon: framed minimalist wall art
[Search] Amazon: minimalist framed art
[Search] Amazon: minimalist wall decor framed
[Search] Amazon: simple framed wall art
[Search] Burrow: framed minimalist wall art
[Search] 

In [ ]:
import pandas as pd
df = pd.read_csv("/content/browserbase_results.csv")

print(f"Total: {len(df)}")
print(f"Queries: {df['query_id'].nunique()} / 15\n")

print("Per query counts:")
print(df['query_id'].value_counts().sort_index())

print(f"\nho_05 sample:")
print(df[df['query_id']=='ho_05'][['title', 'price_text', 'relevance_score']].to_string(index=False))

# Re-check image quality
real_images = df['image_url'].astype(str).str.startswith('http').sum()
print(f"\nReal image URLs: {real_images}/{len(df)} ({100*real_images/len(df):.0f}%)")

Total: 113
Queries: 15 / 15

Per query counts:
query_id
dr_01    13
dr_02     6
dr_03     5
dr_04     4
dr_05    15
ho_01     3
ho_02     3
ho_03    15
ho_04     2
ho_05    14
lr_01     5
lr_02     7
lr_03    11
lr_04     3
lr_05     7
Name: count, dtype: int64

ho_05 sample:
                                                       title price_text  relevance_score
    "Brush Stroke" Framed Textile Art by Minted for West Elm      $ 520              100
                          Layered Paper Composition Wall Art      $ 449               95
                              Leaving Spaces Framed Wall Art      $ 449               95
                            Paper Folding IV Framed Wall Art      $ 375               95
                      42 Pressed Scape Abstract Wall Art - 3      $ 325               90
                        Change Framed Wall Art by Dan Hobday      $ 798               90
                  Blue Layered Curves Framed Canvas Wall Art      $ 479               85
           

In [ ]:
from google.colab import files
files.download("/content/browserbase_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>